In [0]:
%run ./../config/00_project_config

In [0]:
%run ./../setup/00_storage_configuration

In [0]:
suppliers_bronze_path = f"{BRONZE_PATH}/suppliers"
suppliers_silver_path = f"{SILVER_PATH}/suppliers"

In [0]:
df_suppliers_bronze = spark.read.format("delta") \
    .load(suppliers_bronze_path)

In [0]:
df_suppliers_bronze.printSchema()

In [0]:
display(df_suppliers_bronze.limit(10))

In [0]:
from pyspark.sql import functions as F

In [0]:
df_suppliers_silver = df_suppliers_bronze \
    .withColumn(
        "supplier_name",
        F.initcap(F.col("supplier_name"))
    )

In [0]:
display(df_suppliers_silver)

In [0]:
%skip
df_suppliers_silver.write.format("delta") \
    .mode("append") \
    .save(suppliers_silver_path)

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forPath(
    spark,
    suppliers_silver_path
)

silver_table.alias("target") \
.merge(
    df_suppliers_silver.alias("source"),
    "target.supplier_id = source.supplier_id"
) \
.whenMatchedUpdate(
    set = {
        "supplier_name": "source.supplier_name",
        "contact_email": "source.contact_email",
        "phone": "source.phone",
        "country": "source.country"
    }
) \
.whenNotMatchedInsertAll() \
.execute()

In [0]:
spark.read.format("delta") \
    .load(suppliers_silver_path) \
    .count()